In [124]:
import pandas as pd
import numpy as np
from datetime import datetime
from transformers import pipeline
import torch

## Loading data

In [125]:
NEWS_PATH = "data_alpacanews\\historical_news_poland_and_bigcorp-01012015-25022026.csv"
NVIDIA_TICKERS_PATH = "data_stooq\\daily\\us\\nasdaq stocks\\2\\nvda.us.txt"

In [126]:
news_df = pd.read_csv(NEWS_PATH)
nvidia_tickers_df = pd.read_csv(NVIDIA_TICKERS_PATH)
# convert DATE column
nvidia_tickers_df["<DATE>"] = pd.to_datetime(nvidia_tickers_df["<DATE>"], format="%Y%m%d")
news_df['created_at'] = pd.to_datetime(news_df['created_at'], utc = True)

In [127]:
news_df.head(2)
# for item in news_df['full_text'].head(5):
#     print(item)
#     print("__________")

,created_at,symbols,headline,summary,full_text,source,url
0,2026-02-26 13:46:10+00:00,"NVDA,SNDK,SSNLF,WDC",Why Are SanDisk Shares Surging Thursday?,SanDisk Corp (SNDK) is bouncing Thursday after...,Why Are SanDisk Shares Surging Thursday?. SanD...,benzinga,https://www.benzinga.com/trading-ideas/movers/...
1,2026-02-26 13:42:28+00:00,NVDA,"Live On CNBC, NVIDIA CEO Jensen Huang Discusse...",NaN,"Live On CNBC, NVIDIA CEO Jensen Huang Discusse...",benzinga,https://www.benzinga.com/news/26/02/50886686/l...


In [128]:
nvidia_tickers_df.head(2)

,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
0,NVDA.US,D,1999-01-22,0,0.040127,0.044710,0.035533,0.037602,740416863,0
1,NVDA.US,D,1999-01-25,0,0.040586,0.041963,0.037602,0.041499,139413824,0


## Utility function to get ticker data per date

In [129]:
def grab_tickers_around_date(date : str | datetime, 
                             tickers_df : pd.DataFrame, 
                             symbols : str | list | None = None, 
                             days_before : int = 7, 
                             days_after : int =7) -> pd.DataFrame:
    """
    Get ticker price rows around a given date.

    Parameters
    ----------
    date : str | datetime
        Event/news date
    tickers_df : pd.DataFrame
        DataFrame containing price data with columns like ['TICKER','DATE',...]
    symbols : str | list | None
        Comma-separated tickers or list of tickers (e.g. "NVDA,SNDK")
    days_before : int
    days_after : int

    Returns
    -------
    pd.DataFrame
    """

    df = tickers_df.copy()

    date = pd.to_datetime(date)
    start = date - pd.Timedelta(days=days_before)
    end = date + pd.Timedelta(days=days_after)

    # date filtering
    df = df[(df["<DATE>"] >= start) & (df["<DATE>"] <= end)]

    # ticker filtering (optional)
    if symbols is not None:
        if isinstance(symbols, str):
            symbols = [s.strip() for s in symbols.split(",")]

        df = df[df["<TICKER>"].str.replace(".US", "", regex=False).isin(symbols)]

    return df

In [130]:
grab_tickers_around_date(
    "2026-02-2",
    nvidia_tickers_df,
    days_before=3,
    days_after=3
)

,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>
6796,NVDA.US,D,2026-01-30,0,191.210,194.490,189.47,191.13,179489463,0
6797,NVDA.US,D,2026-02-02,0,187.200,190.300,184.88,185.61,165794054,0
6798,NVDA.US,D,2026-02-03,0,186.240,186.270,176.23,180.34,204019589,0
6799,NVDA.US,D,2026-02-04,0,179.460,179.580,171.91,174.19,207014116,0
6800,NVDA.US,D,2026-02-05,0,174.925,176.815,171.03,171.88,206312890,0


## Sentiment analysis of news

In [131]:
news_df = pd.read_csv("news_with_sentiment.csv")
news_df['created_at'] = pd.to_datetime(news_df['created_at'], utc = True)

C:\Users\MatG\AppData\Local\Temp\ipykernel_23924\3538809142.py:1: DtypeWarning: Columns (0: source) have mixed types. Specify dtype option on import or set low_memory=False.
  news_df = pd.read_csv("news_with_sentiment.csv")


In [132]:
# Check device
device = 0 if torch.cuda.is_available() else -1

# Load sentiment pipeline
sentiment_pipeline = pipeline(
    task="text-classification",
    model="ProsusAI/finbert",   # financial-domain BERT: positive / negative / neutral
    tokenizer="ProsusAI/finbert",
    device=-1,                  # -1 = CPU; set to 0 for GPU (CUDA)
    truncation=True,
    max_length=512,
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 50238.67it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [133]:
def get_sentiment(text: str) -> tuple[str, float]:
    if not isinstance(text, str) or not text.strip():
        return "neutral", 0.0
    result = sentiment_pipeline(text)[0]
    return result["label"].lower(), round(result["score"], 4)

In [134]:
if 'sentiment' not in news_df.columns:
    sentiments = []
    scores = []

    for i, text in enumerate(news_df["full_text"]):
        label, score = get_sentiment(text)
        sentiments.append(label)
        scores.append(score)

        if i % 100 == 0:
            print(f"[{i}/{len(news_df)}] done")

    news_df["sentiment"] = sentiments
    news_df["sentiment_score"] = scores

    news_df[["full_text", "sentiment", "sentiment_score"]].head(3)

In [135]:
news_df.to_csv("news_with_sentiment.csv") # Saving just to not loose 1h of progress

In [136]:
news_df.head(2)

,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,created_at,symbols,headline,summary,full_text,source,url,sentiment,sentiment_score
0,0,0,0,2026-02-26 13:46:10+00:00,"NVDA,SNDK,SSNLF,WDC",Why Are SanDisk Shares Surging Thursday?,SanDisk Corp (SNDK) is bouncing Thursday after...,Why Are SanDisk Shares Surging Thursday?. SanD...,benzinga,https://www.benzinga.com/trading-ideas/movers/...,positive,0.9275
1,1,1,1,2026-02-26 13:42:28+00:00,NVDA,"Live On CNBC, NVIDIA CEO Jensen Huang Discusse...",NaN,"Live On CNBC, NVIDIA CEO Jensen Huang Discusse...",benzinga,https://www.benzinga.com/news/26/02/50886686/l...,neutral,0.9317


## Calculaing avg change of stock vs avg change after news

In [137]:
nvidia_tickers_df["daily_return"] = nvidia_tickers_df["<CLOSE>"].pct_change() * 100

In [138]:
nvidia_tickers_df.head(5)

,<TICKER>,<PER>,<DATE>,<TIME>,<OPEN>,<HIGH>,<LOW>,<CLOSE>,<VOL>,<OPENINT>,daily_return
0,NVDA.US,D,1999-01-22,0,0.040127,0.044710,0.035533,0.037602,740416863,0,NaN
1,NVDA.US,D,1999-01-25,0,0.040586,0.041963,0.037602,0.041499,139413824,0,10.364663
2,NVDA.US,D,1999-01-26,0,0.041963,0.042876,0.037839,0.038287,93919355,0,-7.740650
3,NVDA.US,D,1999-01-27,0,0.038512,0.039433,0.036230,0.038287,67279765,0,0.000000
4,NVDA.US,D,1999-01-28,0,0.038287,0.038512,0.037839,0.038058,62320835,0,-0.596811


In [176]:
def compute_ticker_change_no_news(tickers_df: pd.DataFrame) -> pd.DataFrame:
    return (
        tickers_df.groupby("<TICKER>")["daily_return"]
        .agg(
            baseline_mean="mean",
            baseline_std="std",
            baseline_n="count",
        )
        .reset_index()
        .rename(columns={"<TICKER>": "ticker"})
        .assign(ticker=lambda df: df["ticker"].str.replace(".US", "", case=False, regex=False))
    )

In [177]:
ticker_change_no_news_df = compute_ticker_change_no_news(nvidia_tickers_df)
ticker_change_no_news_df

,ticker,baseline_mean,baseline_std,baseline_n
0,NVDA,0.195172,3.75266,6810


In [178]:
def get_next_trading_return(
    news_date: pd.Timestamp,
    ticker_df: pd.DataFrame,        # already filtered to one ticker, sorted by date
    max_days_lookahead: int = 5,    # skip if no trading day found within N calendar days
) -> float | None:
    future = ticker_df[ticker_df["<DATE>"] > news_date]
    if future.empty:
        return None
    next_row = future.iloc[0]
    # reject if too far ahead (e.g. long holiday gap)
    if (next_row["<DATE>"] - news_date).days > max_days_lookahead:
        return None
    return next_row["daily_return"]

In [186]:
SENTIMENT_SCORE = {"positive": 1, "negative": -1, "neutral": 0}

def sentiment_bucket(x):
    if x > 0.2:   return "positive"
    if x < -0.2:  return "negative"
    return "neutral"

def match_news_to_returns(
    news_df: pd.DataFrame,
    tickers_df: pd.DataFrame,
    news_date_col: str = "created_at",
    news_ticker_col: str = "symbols",
    sentiment_col: str = "sentiment",
    min_articles: int = 20,
) -> pd.DataFrame:
    news_df = news_df.copy()

    news_df[news_date_col] = (
        pd.to_datetime(news_df[news_date_col], utc=True)
        .dt.tz_localize(None)
    )

    ticker_name = tickers_df["<TICKER>"].iloc[0].replace(".US", "")

    mask = (
        news_df[news_ticker_col]
        .str.replace(" ", "")
        .str.contains(ticker_name, case=False, na=False)
    )
    ticker_news = news_df[mask].copy()

    if len(ticker_news) < min_articles:
        print(f"Not enough articles for {ticker_name}: {len(ticker_news)} < {min_articles}")
        return pd.DataFrame()

    ticker_news["sentiment_numeric"] = ticker_news[sentiment_col].map(SENTIMENT_SCORE)
    ticker_news["date_only"] = ticker_news[news_date_col].dt.normalize()

    daily_sentiment = (
        ticker_news.groupby("date_only")
        .agg(
            sentiment_sum=("sentiment_numeric", "sum"),    # net sentiment: +3 means 3 more positive than negative
            sentiment_mean=("sentiment_numeric", "mean"),  # avg: normalised for days with many articles
            article_count=("sentiment_numeric", "count"),
            full_text=("full_text", lambda x: " | ".join(x.dropna().astype(str))),
        )
        .reset_index()
        .rename(columns={"date_only": "news_date"})
    )

    records = []
    for _, row in daily_sentiment.iterrows():
        news_date = row["news_date"]
        if hasattr(news_date, "tzinfo") and news_date.tzinfo is not None:
            news_date = news_date.replace(tzinfo=None)

        ret = get_next_trading_return(news_date, tickers_df)
        if ret is None:
            continue
        records.append({
            "ticker": ticker_name,
            "news_date": news_date,
            "sentiment_sum": row["sentiment_sum"],
            "sentiment_mean": row["sentiment_mean"],
            "article_count": row["article_count"],
            "full_text": row["full_text"],
            "sentiment": sentiment_bucket(row["sentiment_sum"]),
            "next_day_return": ret,
        })

    return pd.DataFrame(records)

In [187]:
news_returns = match_news_to_returns(news_df, nvidia_tickers_df)
news_returns

,ticker,news_date,sentiment_sum,sentiment_mean,article_count,full_text,sentiment,next_day_return
0,NVDA,2015-01-05,0,0.000000,1,S&P 500 Stocks With Most Active Options Today:...,neutral,-3.041602
1,NVDA,2015-01-06,0,0.000000,1,"Will NVIDIA's Tegra X1 Dethrone PlayStation 4,...",neutral,-0.267082
2,NVDA,2015-01-09,0,0.000000,1,Barclays: Connected Cars Take Center Stage At ...,neutral,-1.272937
3,NVDA,2015-01-23,0,0.000000,1,Morgan Stanley comments on Mobileye N.V..,neutral,-0.510116
4,NVDA,2015-01-29,0,0.000000,1,Option Alert: Nvidia Feb $20 Call; 8189 Contra...,neutral,-2.940551
...,...,...,...,...,...,...,...,...
2505,NVDA,2026-02-15,1,1.000000,1,Elon Musk Says Tesla Will Have 'Largest Fleet'...,positive,1.181555
2506,NVDA,2026-02-16,1,0.250000,4,10 Information Technology Stocks With Whale Al...,positive,1.181555
2507,NVDA,2026-02-17,0,0.000000,22,David Tepper's Appaloosa Ups Micron Stake By 2...,neutral,1.627291
2508,NVDA,2026-02-18,4,0.190476,21,"'Dr. Doom' Nouriel Roubini Changes His Tune, S...",positive,-0.042558


In [189]:
def compute_news_impact(
    news_returns_df: pd.DataFrame,
    baseline_df: pd.DataFrame,      # output of compute_ticker_change_no_news()
) -> pd.DataFrame:

    summary = (
        news_returns_df.groupby(["ticker", "sentiment"])["next_day_return"]
        .agg(
            news_mean="mean",
            news_std="std",
            news_n="count",
        )
        .reset_index()
    )

    summary = summary.merge(baseline_df, on="ticker", how="left")

    summary["return_diff"] = summary["news_mean"] - summary["baseline_mean"]

    return summary.sort_values(["ticker", "sentiment"]).reset_index(drop=True)


baseline_df = compute_ticker_change_no_news(nvidia_tickers_df)
impact_df = compute_news_impact(news_returns, baseline_df)
impact_df

,ticker,sentiment,news_mean,news_std,news_n,baseline_mean,baseline_std,baseline_n,return_diff
0,NVDA,negative,0.396841,3.405021,654,0.195172,3.75266,6810,0.201669
1,NVDA,neutral,0.263679,3.304436,748,0.195172,3.75266,6810,0.068508
2,NVDA,positive,0.265233,3.005865,1108,0.195172,3.75266,6810,0.070061


## Company different than nvidia